# A1.2 · Designing the agent control plane

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

---

**Risk.** Controls assumed to live 'in the agent' are advisory, not enforced.

**Control.** Identity fabric → agentic gateway → policy decision point → sandboxed runtime → audited action plane.

**This lab.** Stand up the whole reference control plane locally.

| | |
|---|---|
| Open-source tooling | kind, SPIRE, agentgateway, OPA |
| Open-weight models | Llama 3.3 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A1.2"))

The control plane is the only place a design decision can actually bind. Here it is as three composable policies — the same shape you get from OPA, Kyverno and a service mesh, minus the YAML.

In [ ]:
from cybercommons import sandbox

box = sandbox.Sandbox(
    egress=sandbox.EgressPolicy(allow_hosts={"api.github.com"},
                                allow_suffixes={".internal.example"}),
    paths=sandbox.PathGuard(workspace="/work"),
    tools=sandbox.ToolPolicy(allow={"read_file", "search_code", "http_get"},
                             require_approval={"write_file", "post_comment"},
                             deny={"delete_repo", "rotate_secrets"}))

for tool, target, approved in [
    ("read_file",  "/work/src/app.py", False),
    ("read_file",  "/work/../../root/.ssh/id_rsa", False),
    ("http_get",   "https://api.github.com/repos/x/y", False),
    ("http_get",   "http://169.254.169.254/latest/meta-data/", False),
    ("write_file", "/work/out.txt", False),
    ("write_file", "/work/out.txt", True),
    ("delete_repo", "", True),
]:
    print(box.call(tool, target, approved))

print("\n", box.summary())

Every line is a decision with a reason attached. A control plane whose denials you cannot explain is a control plane you cannot tune — and an untunable control gets switched off the first time it blocks something legitimate.

### Expect

Reads inside the workspace and calls to the allowlisted host succeed. The traversal, the metadata address, the ungated write and the denied tool are all refused — the denied write succeeds only once approval is presented. Note that `delete_repo` is refused *even with approval*.

### Your turn

Add a fourth lever: a rate limit. What is the right unit — calls per minute, or state-changing calls per minute? Only one of them bounds damage.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A1.2.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*